# Video ProcessingCovers webcam capture, network streams, screen recording, video-file playback, and saving processed video.

## Import libraries

In [1]:
# import Open CV library
import cv2

import matplotlib.pyplot as plt
import numpy as np

## Webcam
OpenCV allows direct access to a webcam (or any video capture device) using the cv2.VideoCapture() class. Each frame from the webcam can be read in a loop, processed (e.g., grayscale conversion, edge detection), and displayed in real time.

- 1-Initialize webcam: <code></code> 
    - <code>cap = cv2.VideoCapture(0)</code>
    - 0 → default webcam.
    - 1, 2, ... → additional cameras if multiple are connected.
- 2-Read frames:
    - Use <code>ret, frame = cap.read()</code> inside a loop.
    - <code>ret</code> is a boolean indicating success, <code>frame</code> is the captured image.
- 3-Process/display frames:
    - Frames can be converted, filtered, or used in detection algorithms.
    - Display with <code>cv2.imshow()</code>.
- 4-Exit loop: 
    - Use <code>cv2.waitKey()</code> to listen for key presses (e.g., ESC to quit).
- 5-Release resources:
    - Always call <code>cap.release()</code> and <code>cv2.destroyAllWindows()</code> after finishing.


In [ ]:
webcam = cv2.VideoCapture(0)

while True:
    _,frame = webcam.read()
    cv2.imshow('webcam',frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27: # Esc key for closing the window
        break

webcam.release()
cv2.destroyAllWindows()

frame.shape

(480, 640, 3)

## Online Stream
cv2.VideoCapture() can open any video stream source:
- RTSP / HTTP streams (CCTV, IP cameras, online streams)
- Some online video URLs (depending on codec/format and FFmpeg support)

I found this url from this [GitHub repository](https://github.com/fury999io/public-ip-cams?tab=readme-ov-file)

In [14]:
url = "http://213.3.30.80:6001/axis-cgi/mjpg/video.cgi"
stream = cv2.VideoCapture(url)

while True:
    _,frame = stream.read()
    cv2.imshow('stream',frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27: # Esc key for closing the window
        break

stream.release()
cv2.destroyAllWindows()

## Screenrecord
OpenCV can also be used for screen recording by capturing the desktop frames and saving them as a video. While OpenCV cannot directly capture the screen, it can record frames obtained from libraries like PyAutoGUI or mss.

- Capture the screen → using a library like <code>mss</code>(faster) or <code>PyAutoGUI</code>.
- Convert the captured frame → to a NumPy array compatible with OpenCV.
- Save frames to video → using <code>cv2.VideoWriter</code>.

In [ ]:
!pip install mss

In [15]:
import cv2
import numpy as np
import mss

# Define screen capture dimensions (full screen or region)
monitor = {"top": 0, "left": 0, "width": 800, "height": 600}

# Define VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter("screen_record.avi", fourcc, 20.0, (monitor["width"], monitor["height"]))

with mss.mss() as sct:
    while True:
        # Capture screen
        img = np.array(sct.grab(monitor))

        # Convert BGRA to BGR
        frame = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

        # Write to video
        out.write(frame)

        # Display live preview
        cv2.imshow("Screen Recording", frame)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
            break

out.release()
cv2.destroyAllWindows()


## Video File
OpenCV uses the <code>cv2.VideoCapture()</code> class not only for webcams but also for reading video files. Each frame can be processed in a loop and displayed sequentially, giving the effect of video playback.

- 1-Initialize: <code></code> 
    - <code>videofile = cv2.VideoCapture("file path")</code>
- 2-Read frames:
    - Use <code>ret, frame = cap.read()</code> inside a loop.
    - <code>ret = False</code> means the video has ended or cannot be read., <code>frame</code> is the captured frame.
- 3-Process/display frames:
    - Frames can be converted, filtered, or used in detection algorithms.
    - Display with <code>cv2.imshow()</code>.
- 4-Exit loop: 
    - <code>cv2.waitKey(delay)</code> controls playback speed — the delay (in ms) is usually set according to the video’s FPS.
    - Break if video ended <code>ret = False</code>
    - Use <code>cv2.waitKey()</code> to listen for key presses (e.g., ESC to quit).
- 5-Release resources:
    - Always call <code>cap.release()</code> and <code>cv2.destroyAllWindows()</code> after finishing.


In [ ]:
videofile = cv2.VideoCapture("videos\little-my.mp4")
print(f'Height : {videofile.get(cv2.CAP_PROP_FRAME_HEIGHT)}')
print(f'Width : {videofile.get(cv2.CAP_PROP_FRAME_WIDTH)}')
print(f'FPS : {videofile.get(cv2.CAP_PROP_FPS)}')

fps = videofile.get(cv2.CAP_PROP_FPS)
# Control playback speed to match the real FPS
delay = int(1000 / fps)

while True:
    ret,frame = videofile.read()
    if not ret :
        break
    
    cv2.imshow('webcam',frame)
    key = cv2.waitKey(delay) & 0xFF
    if key == 27 : # Esc key for closing the window
        break

videofile.release()
cv2.destroyAllWindows()


Height : 166.0
Width : 166.0
FPS : 16.0


## Saving Videos
To write frames (to a new file) OpenCV uses <code>cv2.VideoWriter</code>.

<code>cv2.VideoWriter(filename, fourcc, fps, frame_size)</code> : <code></code>
- <code>filename</code> → name of the output file (e.g., "output.avi").
- <code>fourcc</code> → codec used to compress the video. common codecs:
    - <code>cv2.VideoWriter_fourcc(*'XVID')</code> for <code>.avi</code>
    - <code>cv2.VideoWriter_fourcc(*'mp4v')</code> for <code>.mp4</code>
- <code>fps</code> → frames per second (should match input for smooth playback).
- <code>frame_size</code> → (width, height) of the video frames. Must **exactly** match the size of the frames being written.

After creating the writer, you call <code>writer.write(frame)</code> for each frame you want to save. Finally, call <code>writer.release()</code> when done.


### Save from Webcam

In [ ]:
webcam = cv2.VideoCapture(0)

width  = int(webcam.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(webcam.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'XVID')
# Create writer
webcam_output = cv2.VideoWriter("webcam_output.avi", fourcc, fps, (width, height))

while True:
    # Read
    ret,frame = webcam.read()
    if not ret :
        break

    # Save
    webcam_output.write(frame)

    # Display webcam
    cv2.imshow('webcam',frame)

    key = cv2.waitKey(delay) & 0xFF
    if key == 27 : # Esc key for closing the window
        break

webcam.release()
webcam_output.release()
cv2.destroyAllWindows()


### Save from File
and convert videofile to grayscale

In [ ]:
cap = cv2.VideoCapture("videos/snufkin.mp4")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'XVID')
video_output = cv2.VideoWriter("grayscale_output.avi", fourcc, fps, (width, height),isColor=False) # isColor=False because output is grayscale

while True:
    # Read
    ret,frame = cap.read()
    if not ret :
        break

    # Convert frame to GRAY
    frame = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

    # Write
    video_output.write(frame)

    # Display grayscale video
    cv2.imshow('grayscale video',frame)
    key = cv2.waitKey(int(1000/fps)) & 0xFF
    if key == 27 : # Esc key for closing the window
        break

cap.release()
video_output.release()
cv2.destroyAllWindows()